# CAMRA — Notebook 1: Dataset creation

Builds the execution-level dataset from the raw CIC-YNU-IoT-MAL-26 parquet files. For each architecture and modality, records are aggregated per file hash into statistical features; the three modalities are then inner-joined on the hash, so that each row is one execution with its PCAP, SAR, and STRACE features. The result is saved for `02_camra_pipeline.ipynb`.

Set `DATA_ROOT` and `OUT_DIR` in the configuration cell.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ═══════════════════════════════════════════════════════════
import os, gc, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Set

from scipy.stats import skew as scipy_skew

import pyarrow as pa
import pyarrow.parquet as pq

try:
    import scienceplots
    plt.style.use(['science', 'ieee', 'no-latex'])
    # scienceplots' 'science' style turns on text.usetex, which silently requires a FULL local LaTeX toolchain (incl. type1cm/type1ec) - common on a 
    # personal machine, often MISSING on Kaggle/CI images. Importing the package never fails in that case; only the first real render does, deep inside 
    # savefig. Probe it here with a throwaway plot so we fail fast and fall back cleanly instead of crashing hours into a run.
    _fig = plt.figure()
    plt.plot([0, 1], [0, 1]); plt.title("probe")
    _fig.savefig(Path("/tmp") / "_style_probe.png")
    plt.close(_fig)
except Exception as e:
    print(f"[INFO] scienceplots/LaTeX style unavailable ({type(e).__name__}) - using default matplotlib style")
    plt.style.use('default')
    matplotlib.rcParams['text.usetex'] = False

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

## Configuration

Data paths, schema overrides, and curation settings.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION
# ═══════════════════════════════════════════════════════════

_KAGGLE_CANDIDATES = [
    # Path("/kaggle/input/datasets/ashmitcajla/cic-2026/cic-ynu-iot-mal-26"),
    # Path("/kaggle/input/cic-ynu-iot-mal-26"),
    # Path("/kaggle/input/cic-2026"),
]
_LOCAL_ROOT = Path("../data/cic26")
DATA_ROOT   = next((p for p in _KAGGLE_CANDIDATES if p.exists()), _LOCAL_ROOT)
print(f"[INFO] DATA_ROOT resolved to: {DATA_ROOT}")

OUT_DIR = Path("../data/cleaned")
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / "plots").mkdir(exist_ok=True)
(OUT_DIR / "reports").mkdir(exist_ok=True)

ARCHS       = ["ARM", "MIPS", "MIPSEL", "x86"]
MODALITIES  = ["PCAP", "SAR", "STRACE"]

# --- schema: confirmed common columns across all 3 modality files ----------
# The join key ("hash") and target column are known column names in this dataset; candidates lists below are only a safety net in case a specific
# file uses different capitalization, or you point this at a different export.
HASH_COL_OVERRIDE   : Optional[str] = None   # set e.g. "hash" to force it
TARGET_COL_OVERRIDE : Optional[str] = None   # set e.g. "MalwareFamily" to force it
ARCH_COL_OVERRIDE   : Optional[str] = None   # set e.g. "Arch" to force it

HASH_COL_CANDIDATES   = ["hash", "filehash", "file_hash", "sha256", "md5", "sample_id", "sampleid", "sample"]
TARGET_COL_CANDIDATES = ["malwarefamily", "label", "family", "class", "target"]
ARCH_COL_CANDIDATES   = ["arch", "architecture", "cpu_arch"]

DROP_LABELS = {"Unknown"}  # rows with this label are excluded before anything else

# --- alignment / aggregation ------------------------------------------------
LOW_VARIANCE_MAX_TRACK = 2   # values tracked per column to detect "constant"

# For every raw numeric column, also compute a richer set of aggregate
# statistics beyond plain mean/std/min/max:
#   - variance          (std^2 - same info as std, but ANOVA F-selection can score it differently since squaring reshapes the distribution)

#   - cv (coefficient of variation, std / |mean|) - relative dispersion, scale-invariant: separates "erratic relative to its own level" from "erratic in 
#       absolute terms", which mean/std alone can't distinguish

#   - logmean, logstd   (mean/std of log1p of the non-negative values) -
#                         geometric-mean-like, robust to the rare-burst
#                         outliers confirmed on real data (a SAR interrupt
#                         counter with median 0 but max in the thousands)
#   - geomean           (expm1(logmean), i.e. the geometric mean back in the
#                         ORIGINAL units) - same robustness as logmean, but
#                         on the raw column's own scale
# min/max are NOT log-companioned - they're single observed extremes, not
# aggregates an outlier can distort, so a log version adds no new robustness
# (Notebook 2's per-column skew-driven log-transform already covers min/max
# individually if a specific one turns out to need it).
# All of this uses the same O(unique-hashes) streaming design - variance/cv/
# geomean are cheap derived values from state already being tracked, not
# additional passes over the data. Set False to skip logmean/logstd/geomean
# and keep the smaller column set (mean/std/variance/cv/min/max still apply).
COMPUTE_LOG_SPACE_STATS = True

# Malware families with fewer than this many aligned executions get relabeled
# to "Other" (see merge_rare_labels below) instead of staying as their own
# class. A class this small can't be meaningfully split, trained, or
# evaluated - confirmed on real data: e.g. a 1-sample class is, by
# construction, entirely in exactly one of train/val/test, and a 3-sample
# class checked out as 100% confined to a single architecture, meaning a
# model "learning" it would really just be memorizing one specific execution.
MIN_SAMPLES_PER_CLASS = 5
RARE_LABEL_BUCKET_NAME = "Other"

# Optional hard cap on aligned executions per class, applied AFTER alignment.
# Leave as None to keep every aligned execution (recommended - this is what
# actually fixes the "throwing away 97% of STRACE" problem). Set an int only
# if you need to bound runtime/memory on the full dataset.
POST_ALIGNMENT_PER_CLASS_CAP: Optional[int] = None

# A column can still have a handful of per-execution NaN after alignment
# (e.g. one run's STRACE rows were all null/inf for a field that other runs
# have valid data for). Below this many valid values, imputing with a mean
# isn't trustworthy, so we fall back to 0 instead - see impute_residual_nans.
MIN_VALID_FOR_MEAN_IMPUTE = 10

## Schema resolution helpers

Case-insensitive lookup of the hash, label, and architecture columns, so that files with slightly different column names are handled.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — SCHEMA RESOLUTION HELPERS
# ═══════════════════════════════════════════════════════════

def resolve_column(columns: List[str], candidates: List[str], override: Optional[str] = None) -> Optional[str]:
    """Case-insensitive column-name resolver. `override` wins if given."""
    if override is not None:
        return override if override in columns else None
    lower_map = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def find_modality_path(base: Path, mod: str) -> Optional[Path]:
    for p in [base / f"{mod}.parquet", base / f"{mod.lower()}.parquet", base / f"{mod.upper()}.parquet"]:
        if p.exists():
            return p
    return None


def arch_base_dir(arch: str) -> Optional[Path]:
    for d in [DATA_ROOT / arch, DATA_ROOT / arch.lower(), DATA_ROOT / arch.upper()]:
        if d.exists():
            return d
    return None

## Schema diagnostics

For every architecture × modality file: row count, resolved hash / label / architecture columns, and number of numeric features.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — SCHEMA DIAGNOSTICS (run first, sanity-check the output)
# ═══════════════════════════════════════════════════════════

def schema_diagnostics() -> pd.DataFrame:
    rows = []
    for arch in ARCHS:
        base = arch_base_dir(arch)
        if base is None:
            print(f"[WARN] No directory found for architecture '{arch}' under {DATA_ROOT}")
            continue
        for mod in MODALITIES:
            path = find_modality_path(base, mod)
            if path is None:
                print(f"[WARN] No {mod}.parquet found for architecture '{arch}' under {base}")
                continue
            pf = pq.ParquetFile(path)
            cols = [f.name for f in pf.schema_arrow]
            hash_col   = resolve_column(cols, HASH_COL_CANDIDATES, HASH_COL_OVERRIDE)
            target_col = resolve_column(cols, TARGET_COL_CANDIDATES, TARGET_COL_OVERRIDE)
            arch_col   = resolve_column(cols, ARCH_COL_CANDIDATES, ARCH_COL_OVERRIDE)
            n_numeric  = sum(1 for f in pf.schema_arrow if pa.types.is_integer(f.type) or pa.types.is_floating(f.type))
            rows.append({
                "arch": arch, "modality": mod, "path": str(path),
                "n_rows": pf.metadata.num_rows, "n_row_groups": pf.num_row_groups,
                "n_columns": len(cols), "n_numeric_cols": n_numeric,
                "hash_col_detected": hash_col, "target_col_detected": target_col,
                "arch_col_detected": arch_col,
            })
    report = pd.DataFrame(rows)
    missing = report[report[["hash_col_detected", "target_col_detected"]].isna().any(axis=1)]
    if len(missing):
        print("[WARN] Could not resolve hash/target column for some files - "
              "set HASH_COL_OVERRIDE / TARGET_COL_OVERRIDE above:")
        print(missing[["arch", "modality", "hash_col_detected", "target_col_detected"]])
    return report

## Per-hash aggregation and cross-modality alignment

Detects constant columns, streams each file in row groups (bounded memory) to compute per-hash statistics, and inner-joins PCAP, SAR, and STRACE on the hash, with a coverage report.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — LOW-VARIANCE COLUMN DETECTION (streamed, bounded memory)
# ═══════════════════════════════════════════════════════════

def detect_low_variance_columns(paths, numeric_cols: List[str],
                                max_track: int = LOW_VARIANCE_MAX_TRACK) -> List[str]:
    """Stream row-groups from one or more parquet files, tracking up to `max_track` distinct NON-NULL values per numeric column. A column with
    <=1 distinct value across ALL rows in ALL given files - including the case where every value is null, which counts as 0 distinct values, still
    <=1 - is flagged for removal. We stop tracking a column as soon as it has >1 distinct value, so memory stays bounded regardless of file size.

    Pass a LIST of paths (one per architecture) to decide variance ACROSS architectures combined, rather than per file. This matters because a
    column can be entirely null in one architecture's export while varying normally in another (e.g. a metric that arch's instrumentation never
    populates) - deciding per-file would drop it in that one architecture only, leaving every other architecture with the column, and concatenating
    them later would silently NaN-fill the architecture that dropped it."""
    
    if isinstance(paths, (str, Path)):
        paths = [paths]
    seen: Dict[str, set] = {c: set() for c in numeric_cols}
    active = set(numeric_cols)
    for path in paths:
        if not active:
            break
        pf = pq.ParquetFile(path)
        for i in range(pf.num_row_groups):
            if not active:
                break
            chunk = pf.read_row_group(i, columns=list(active)).to_pandas()
            for c in list(active):
                for v in pd.unique(chunk[c].dropna()):
                    seen[c].add(v)
                    if len(seen[c]) > 1:
                        break
                if len(seen[c]) > 1:
                    active.discard(c)
            del chunk
    return [c for c, s in seen.items() if len(s) <= 1]

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — PER-HASH STREAMING AGGREGATOR
# ═══════════════════════════════════════════════════════════

class HashAggregator:
    """Incrementally builds per-hash (per-execution) mean/std/min/max + record_count for every numeric feature, streaming one row-group chunk at
    a time. Memory is proportional to the number of UNIQUE hashes, not the number of raw rows, so this is safe even for a STRACE file with many
    millions of syscall-level rows: a hash with thousands of raw rows still collapses to a single accumulator entry.

    Mean/variance are combined across chunks using Welford/Chan's parallel algorithm (running mean + M2), NOT the naive `E[X^2] - E[X]^2` formula.
    The naive formula squares raw values before combining them, which overflows to `inf` for large-magnitude columns (byte counters,
    nanosecond timestamps, memory addresses are all common in network/syscall telemetry) and then produces `inf - inf = NaN` - a real failure mode we
    hit on real data, not a hypothetical one. Welford's formula only ever squares DEVIATIONS from a batch's own mean, so it stays numerically
    stable regardless of the raw values' magnitude.

    Both NaN and +/-inf in the raw data are treated as missing for aggregation purposes (`np.isfinite` filters both) - an infinite value
    can't sensibly contribute to a mean/std either. If a hash+column ends up with ZERO valid (finite) raw values, that's recorded as a genuine NaN in
    the output (not silently guessed as 0), so the explicit imputation step in `load_full_aligned_dataset` can handle it visibly and consistently.
    """

    def __init__(self, numeric_cols: List[str], compute_log_stats: bool = True):
        self.numeric_cols = numeric_cols
        self.compute_log_stats = compute_log_stats
        self.n_valid: Dict[str, Dict[str, int]]   = defaultdict(lambda: defaultdict(int))
        self.mean:    Dict[str, Dict[str, float]] = defaultdict(lambda: defaultdict(float))
        self.M2:      Dict[str, Dict[str, float]] = defaultdict(lambda: defaultdict(float))
        self.min:     Dict[str, Dict[str, float]] = defaultdict(lambda: defaultdict(lambda: np.inf))
        self.max:     Dict[str, Dict[str, float]] = defaultdict(lambda: defaultdict(lambda: -np.inf))
        self.count:   Dict[str, int] = defaultdict(int)
        self.labels_seen: Dict[str, Set[str]] = defaultdict(set)
        # Log-space companions (mean/std of log1p(non-negative values)) - geometric-mean-like statistics. On real telemetry, many count-like
        # columns are "almost always 0, occasionally a large burst" (e.g. a SAR interrupt counter with median 0 but max in the thousands); the
        # LINEAR mean/std for such a column is dominated by the rare bursts and barely describes a typical execution. logmean/logstd fix this
        # at the source, using the exact same O(unique hashes) Welford machinery, instead of just log-transforming an already-distorted
        # aggregate after the fact.
        if compute_log_stats:
            self.log_n_valid: Dict[str, Dict[str, int]]   = defaultdict(lambda: defaultdict(int))
            self.log_mean:    Dict[str, Dict[str, float]] = defaultdict(lambda: defaultdict(float))
            self.log_M2:      Dict[str, Dict[str, float]] = defaultdict(lambda: defaultdict(float))

    @staticmethod
    def _welford_merge(n_a: int, mean_a: float, M2_a: float, vals: np.ndarray):
        """O(1)-space combination of a running (count, mean, M2) with a new batch, via Chan et al.'s parallel variance algorithm. Only ever
        squares DEVIATIONS from a batch's own mean (never raw values), so it stays numerically stable regardless of the batch's magnitude."""
        n_b = len(vals)
        if n_b == 0:
            return n_a, mean_a, M2_a
        mean_b = float(vals.mean())
        M2_b   = float(np.sum((vals - mean_b) ** 2))
        if n_a == 0:
            return n_b, mean_b, M2_b
        n_ab  = n_a + n_b
        delta = mean_b - mean_a
        return n_ab, mean_a + delta * n_b / n_ab, M2_a + M2_b + delta ** 2 * n_a * n_b / n_ab

    def update(self, chunk: pd.DataFrame, hash_col: str, target_col: str):
        for h, grp in chunk.groupby(hash_col, sort=False):
            n = len(grp)
            self.count[h] += n
            if target_col in grp.columns:
                self.labels_seen[h].update(grp[target_col].astype(str).unique().tolist())
            for c in self.numeric_cols:
                vals = grp[c].to_numpy(dtype=np.float64)
                vals = vals[np.isfinite(vals)]   # drop NaN AND +/-inf alike
                if len(vals) == 0:
                    continue
                self.n_valid[h][c], self.mean[h][c], self.M2[h][c] = self._welford_merge(
                    self.n_valid[h][c], self.mean[h][c], self.M2[h][c], vals)
                self.min[h][c] = min(self.min[h][c], float(vals.min()))
                self.max[h][c] = max(self.max[h][c], float(vals.max()))

                if self.compute_log_stats:
                    nonneg = vals[vals >= 0]   # log1p undefined below -1; skip negatives
                    if len(nonneg):
                        log_vals = np.log1p(nonneg)
                        (self.log_n_valid[h][c], self.log_mean[h][c],
                         self.log_M2[h][c]) = self._welford_merge(
                            self.log_n_valid[h][c], self.log_mean[h][c], self.log_M2[h][c], log_vals)

    def finalize(self, modality: str) -> pd.DataFrame:
        eps = 1e-8   # guards the coefficient-of-variation divide when mean is ~0
        rows = []
        for h, n in self.count.items():
            row = {"hash": h, f"{modality}__record_count": n}
            for c in self.numeric_cols:
                n_valid = self.n_valid[h][c]
                if n_valid > 0:
                    mean = self.mean[h][c]
                    var  = self.M2[h][c] / n_valid   # population variance, numerically stable
                    # Welford avoids the overflow the naive sum-of-squares formula hits for MODERATELY large values, but a truly
                    # extreme raw value (a corrupt sentinel, a bad sensor
                    # reading) can still overflow the deviation-squared term.
                    # Rather than guess a dataset-specific "too large"
                    # threshold, just check the arithmetic's own output: if
                    # mean/std came out non-finite, treat this hash+column as
                    # missing so it flows through the SAME visible imputation
                    # step as genuine missingness, instead of leaving an inf
                    # in the "clean" dataset that later breaks scaling.
                    std   = float(np.sqrt(var)) if (np.isfinite(var) and var >= 0) else np.nan
                    mean  = mean if np.isfinite(mean) else np.nan
                    var   = var if np.isfinite(var) else np.nan
                    # Coefficient of variation: relative dispersion, scale-
                    # invariant unlike std alone - separates "erratic relative
                    # to its own typical level" from "erratic in absolute
                    # terms", which a raw mean/std pair can't distinguish
                    # (e.g. mean=1000,std=100 and mean=10,std=100 have very
                    # different volatility character despite the same std).
                    cv = (std / (abs(mean) + eps)) if (np.isfinite(mean) and np.isfinite(std)) else np.nan
                    row[f"{modality}__{c}__mean"]     = mean
                    row[f"{modality}__{c}__std"]      = std
                    row[f"{modality}__{c}__variance"] = var
                    row[f"{modality}__{c}__cv"]       = cv
                    row[f"{modality}__{c}__min"]      = self.min[h][c]
                    row[f"{modality}__{c}__max"]      = self.max[h][c]
                else:
                    # This specific execution had zero finite raw values for
                    # this column (while OTHER executions had enough valid
                    # data for the column to survive the global variance
                    # filter). Genuine missingness - left as NaN on purpose,
                    # imputed explicitly and visibly later, not guessed here.
                    row[f"{modality}__{c}__mean"]     = np.nan
                    row[f"{modality}__{c}__std"]      = np.nan
                    row[f"{modality}__{c}__variance"] = np.nan
                    row[f"{modality}__{c}__cv"]       = np.nan
                    row[f"{modality}__{c}__min"]      = np.nan
                    row[f"{modality}__{c}__max"]      = np.nan

                if self.compute_log_stats:
                    log_n_valid = self.log_n_valid[h][c]
                    if log_n_valid > 0:
                        log_mean = self.log_mean[h][c]
                        log_var  = self.log_M2[h][c] / log_n_valid
                        log_std  = float(np.sqrt(log_var)) if (np.isfinite(log_var) and log_var >= 0) else np.nan
                        log_mean = log_mean if np.isfinite(log_mean) else np.nan
                        # Geometric mean, back in the ORIGINAL units (not log
                        # units): expm1 undoes the log1p used to build
                        # log_mean. Distinct from logmean in modeling terms -
                        # it lands on the same scale as the raw column, so it
                        # interacts differently with a linear scaler - while
                        # still being far more outlier-robust than the
                        # arithmetic mean (a handful of huge values pull the
                        # arithmetic mean up a lot; they barely move a
                        # geometric mean).
                        geomean = float(np.expm1(log_mean)) if np.isfinite(log_mean) else np.nan
                        row[f"{modality}__{c}__logmean"] = log_mean
                        row[f"{modality}__{c}__logstd"]  = log_std
                        row[f"{modality}__{c}__geomean"] = geomean
                    else:
                        row[f"{modality}__{c}__logmean"] = np.nan
                        row[f"{modality}__{c}__logstd"]  = np.nan
                        row[f"{modality}__{c}__geomean"] = np.nan
            n_labels = len(self.labels_seen[h])
            row["_label_conflict"] = n_labels > 1
            row["MalwareFamily"] = next(iter(self.labels_seen[h])) if n_labels >= 1 else None
            rows.append(row)
        return pd.DataFrame(rows)


def aggregate_modality_by_hash(path: Path, modality: str, hash_col: str, target_col: str,
                                arch_col: Optional[str],
                                restrict_cols: Optional[List[str]] = None) -> Tuple[pd.DataFrame, Dict]:
    pf = pq.ParquetFile(path)
    schema = pf.schema_arrow
    exclude = {hash_col, target_col, arch_col}
    file_numeric_cols = {f.name for f in schema
                            if (pa.types.is_integer(f.type) or pa.types.is_floating(f.type)) and f.name not in exclude}
    if restrict_cols is not None:
        # Different architectures can have different raw column sets (this is
        # documented for CIC-YNU-IoTMal: SAR/STRACE feature counts differ by
        # arch). Restricting every arch's aggregation to the SAME column set
        # keeps the fused feature space consistent, instead of relying on
        # pandas to silently NaN-fill mismatched columns at concat time.
        # `restrict_cols` (from compute_shared_numeric_columns) has ALREADY
        # had globally-low-variance columns removed, so no need to redo that
        # check per file here.
        numeric_cols = [c for c in restrict_cols if c in file_numeric_cols]
        low_var = []
    else:
        numeric_cols = sorted(file_numeric_cols)
        low_var = detect_low_variance_columns(path, numeric_cols)
        numeric_cols = [c for c in numeric_cols if c not in low_var]

    read_cols = list({hash_col, target_col, *numeric_cols})
    agg = HashAggregator(numeric_cols, compute_log_stats=COMPUTE_LOG_SPACE_STATS)
    total_raw_rows = 0
    for i in range(pf.num_row_groups):
        chunk = pf.read_row_group(i, columns=read_cols).to_pandas()
        chunk.columns = chunk.columns.str.strip()
        if target_col in chunk.columns:
            chunk = chunk[~chunk[target_col].astype(str).isin(DROP_LABELS)]
        total_raw_rows += len(chunk)
        if len(chunk):
            agg.update(chunk, hash_col, target_col)
        del chunk
    gc.collect()

    df_agg = agg.finalize(modality)
    stats = {
        "modality": modality,
        "raw_rows": total_raw_rows,
        "unique_hashes_raw": len(agg.count),
        "dropped_low_variance_cols": low_var,
        "n_features_after_agg": len(numeric_cols) * (9 if COMPUTE_LOG_SPACE_STATS else 6) + 1,
        "per_hash_row_count": dict(agg.count),
    }
    return df_agg, stats

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — CROSS-MODALITY ALIGNMENT (inner join on hash) + COVERAGE REPORT
# ═══════════════════════════════════════════════════════════

def compute_shared_numeric_columns() -> Dict[str, List[str]]:
    """Reads parquet SCHEMAS (cheap) for every arch x modality file to get the intersection of numeric column NAMES available across all 4
    architectures, THEN streams the actual data (still cheap - one pass) to drop any column that is constant or entirely null in at least one
    architecture, checked ACROSS ALL 4 ARCHITECTURES COMBINED (see `detect_low_variance_columns`). Both steps matter: CIC-YNU-IoTMal's own
    published statistics show SAR/STRACE feature *counts* differing by architecture, and even a column present by name in every architecture's
    schema can be entirely null in just one of them. Deciding either of these per-architecture, independently, would let architectures disagree on the
    final column set - and concatenating them later would silently NaN-fill whichever rows came from the architecture that dropped a column
    that others kept."""
    per_mod_sets: Dict[str, List[set]] = {mod: [] for mod in MODALITIES}
    per_mod_paths: Dict[str, List[Path]] = {mod: [] for mod in MODALITIES}
    for arch in ARCHS:
        base = arch_base_dir(arch)
        if base is None:
            continue
        for mod in MODALITIES:
            path = find_modality_path(base, mod)
            if path is None:
                continue
            cols = [f.name for f in pq.ParquetFile(path).schema_arrow]
            hash_col   = resolve_column(cols, HASH_COL_CANDIDATES, HASH_COL_OVERRIDE)
            target_col = resolve_column(cols, TARGET_COL_CANDIDATES, TARGET_COL_OVERRIDE)
            arch_col   = resolve_column(cols, ARCH_COL_CANDIDATES, ARCH_COL_OVERRIDE)
            exclude = {hash_col, target_col, arch_col}
            numeric = {f.name for f in pq.ParquetFile(path).schema_arrow
                       if (pa.types.is_integer(f.type) or pa.types.is_floating(f.type)) and f.name not in exclude}
            per_mod_sets[mod].append(numeric)
            per_mod_paths[mod].append(path)

    shared = {}
    for mod, sets in per_mod_sets.items():
        name_intersection = sorted(set.intersection(*sets)) if sets else []
        low_var = detect_low_variance_columns(per_mod_paths[mod], name_intersection) if name_intersection else []
        shared[mod] = [c for c in name_intersection if c not in low_var]
        if low_var:
            print(f"[INFO] {mod}: dropping {len(low_var)} column(s) constant/all-null in at least one "
                  f"architecture (checked across all architectures combined): "
                  f"{low_var[:10]}{'...' if len(low_var) > 10 else ''}")

    for mod, cols in shared.items():
        print(f"[INFO] Shared, non-constant numeric columns for {mod} across all architectures: {len(cols)}")
    return shared


def build_aligned_arch_dataset(arch: str, shared_numeric_cols: Dict[str, List[str]]) -> Tuple[pd.DataFrame, Dict]:
    """For one architecture: aggregate PCAP/SAR/STRACE per hash, drop any hash
    with an internally-inconsistent label, inner-join the three modalities on
    hash, drop any hash whose label disagrees ACROSS modalities, and return
    the fused per-execution dataframe plus a coverage report quantifying
    exactly how many raw rows / unique executions were retained."""
    base = arch_base_dir(arch)
    if base is None:
        raise FileNotFoundError(f"No directory found for architecture '{arch}' under {DATA_ROOT}")

    per_mod_df, per_mod_stats = {}, {}
    for mod in MODALITIES:
        path = find_modality_path(base, mod)
        if path is None:
            raise FileNotFoundError(f"No {mod}.parquet found for architecture '{arch}' under {base}")
        cols = [f.name for f in pq.ParquetFile(path).schema_arrow]
        hash_col   = resolve_column(cols, HASH_COL_CANDIDATES, HASH_COL_OVERRIDE)
        target_col = resolve_column(cols, TARGET_COL_CANDIDATES, TARGET_COL_OVERRIDE)
        arch_col   = resolve_column(cols, ARCH_COL_CANDIDATES, ARCH_COL_OVERRIDE)
        if hash_col is None or target_col is None:
            raise ValueError(f"Could not resolve hash/target column in {path}. "
                              f"Set HASH_COL_OVERRIDE/TARGET_COL_OVERRIDE. Columns seen: {cols}")
        df_agg, stats = aggregate_modality_by_hash(path, mod, hash_col, target_col, arch_col,
                                                    restrict_cols=shared_numeric_cols.get(mod))
        per_mod_df[mod] = df_agg
        per_mod_stats[mod] = stats

    # drop hashes with a label conflict WITHIN a single modality's own raw rows
    for mod in MODALITIES:
        n_before = len(per_mod_df[mod])
        per_mod_df[mod] = per_mod_df[mod][~per_mod_df[mod]["_label_conflict"]].drop(columns=["_label_conflict"])
        per_mod_stats[mod]["dropped_internal_label_conflicts"] = n_before - len(per_mod_df[mod])

    # inner join on hash: keep only executions present (with a clean label) in ALL 3 modalities
    merged = per_mod_df["PCAP"].merge(
        per_mod_df["SAR"], on="hash", how="inner", suffixes=("", "_sar")
    ).merge(
        per_mod_df["STRACE"], on="hash", how="inner", suffixes=("", "_strace")
    )

    # drop hashes whose label disagrees ACROSS modalities (data-quality guard)
    label_cols = [c for c in merged.columns if c == "MalwareFamily" or c.startswith("MalwareFamily_")]
    agree_mask = merged[label_cols].nunique(axis=1) == 1
    n_cross_conflict = int((~agree_mask).sum())
    merged = merged[agree_mask].copy()
    merged = merged.drop(columns=[c for c in label_cols if c != "MalwareFamily"])
    merged.insert(0, "Arch", arch)

    # NOTE: post-aggregation low-variance filtering (a column can become
    # constant only AFTER per-hash aggregation, e.g. a modality with ~1 raw
    # row per execution makes every std/min/max column degenerate) is
    # deliberately NOT done here, per architecture. Doing it per-arch would
    # let each architecture end up with a DIFFERENT surviving column set, and
    # concatenating frames with mismatched columns later would silently
    # reintroduce NaNs for whichever archs happened to drop a given column
    # (pandas fills the union of columns with NaN). That filter is applied
    # once, globally, in `load_full_aligned_dataset` after concatenation.

    # ---- coverage report ----
    coverage = {"arch": arch, "cross_modality_label_conflicts_dropped": n_cross_conflict}
    aligned_hashes = set(merged["hash"])
    for mod in MODALITIES:
        st = per_mod_stats[mod]
        raw_unique = st["unique_hashes_raw"]
        matched = len(aligned_hashes & set(st["per_hash_row_count"].keys()))
        row_total = st["raw_rows"]
        row_matched = sum(cnt for h, cnt in st["per_hash_row_count"].items() if h in aligned_hashes)
        coverage[mod] = {
            "raw_rows": row_total,
            "unique_hashes_raw": raw_unique,
            "unique_hashes_matched": matched,
            "execution_retention_pct": 100.0 * matched / raw_unique if raw_unique else 0.0,
            "row_coverage_pct": 100.0 * row_matched / row_total if row_total else 0.0,
            "dropped_low_variance_cols": st["dropped_low_variance_cols"],
            "n_features_after_agg": st["n_features_after_agg"],
            "dropped_internal_label_conflicts": st["dropped_internal_label_conflicts"],
        }
    coverage["n_aligned_executions"] = len(merged)
    return merged, coverage


def print_coverage_report(coverage_list: List[Dict]):
    print(f"\n{'Arch':>8} | {'Modality':>8} | {'RawRows':>10} | {'UniqHash':>9} | "
          f"{'Matched':>8} | {'ExecRet%':>9} | {'RowCov%':>8}")
    print("-" * 78)
    for cov in coverage_list:
        for mod in MODALITIES:
            c = cov[mod]
            print(f"{cov['arch']:>8} | {mod:>8} | {c['raw_rows']:>10,} | {c['unique_hashes_raw']:>9,} | "
                  f"{c['unique_hashes_matched']:>8,} | {c['execution_retention_pct']:>8.1f}% | "
                  f"{c['row_coverage_pct']:>7.1f}%")
        print(f"{'':>8} | {'ALIGNED':>8} | {'':>10} | {'':>9} | {cov['n_aligned_executions']:>8,} | "
              f"cross-modal label conflicts dropped: {cov['cross_modality_label_conflicts_dropped']}")
        print("-" * 78)

## Build the aligned dataset

Runs the aggregation and alignment for all architectures, drops hashes with inconsistent labels, and merges families with fewer than five executions into `Other`.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — BUILD FULL ALIGNED DATASET
# ═══════════════════════════════════════════════════════════

def impute_residual_nans(full: pd.DataFrame, min_valid_for_mean: int = 10) -> pd.DataFrame:
    """Fills any NaN remaining in aggregated feature columns after the
    constant/all-null column drop above. Residual NaN at this point means a
    SPECIFIC execution had zero finite raw values for a column that other
    executions DID have valid data for (e.g. one malware run's STRACE rows
    all happened to be null/inf for a particular field) - genuine per-row
    missingness, not a whole-column data problem.

    Policy: impute with that column's own mean (computed from its valid,
    non-NaN values across the aligned dataset) when at least
    `min_valid_for_mean` valid values are available to make that mean
    trustworthy; fall back to 0 when there are too few (or zero) valid values
    to estimate a mean at all. Every imputed column is logged with how many
    rows were affected and which strategy was used - never silent.

    Note: this runs on the FULL aligned dataset, before Notebook 2's
    train/val/test split, so the imputed value for any one row is influenced
    by the column's distribution across ALL rows (including ones that will
    later land in val/test). This is a mild, standard data-cleaning step
    (filling a small number of missing cells with a population statistic),
    not a supervised fit like scaling or feature selection - but if you need
    strictly train-only imputation, leave these as NaN here instead and add
    a train-fit imputer in Notebook 2 alongside the scaler.
    """
    feat_cols = [c for c in full.columns if "__" in c]
    nan_counts = full[feat_cols].isna().sum()
    affected = nan_counts[nan_counts > 0]
    if affected.empty:
        return full

    print(f"\n[INFO] Imputing residual NaN in {len(affected)} column(s) "
          f"(per-execution missingness, not whole-column):")
    full = full.copy()
    for col, n_missing in affected.items():
        valid = full[col].dropna()
        if len(valid) >= min_valid_for_mean:
            fill_value = float(valid.mean())
            strategy = f"mean={fill_value:.4g} (from {len(valid)} valid values)"
        else:
            fill_value = 0.0
            strategy = f"0.0 (only {len(valid)} valid value(s), below min_valid_for_mean={min_valid_for_mean})"
        full[col] = full[col].fillna(fill_value)
        print(f"    {col}: {n_missing} row(s) imputed with {strategy}")
    return full


def merge_rare_labels(full: pd.DataFrame, min_samples: int = MIN_SAMPLES_PER_CLASS,
                       bucket_name: str = RARE_LABEL_BUCKET_NAME) -> pd.DataFrame:
    """Relabels any MalwareFamily with fewer than `min_samples` aligned
    executions to `bucket_name`. A handful of samples can't be meaningfully
    split into train/val/test (a 1-sample class is, by construction, in only
    one split ever), can't support a trustworthy per-class F1, and - checked
    directly on real data - such tiny classes turned out to be near-entirely
    confined to a single architecture, meaning a model "learning" them would
    really just be memorizing one specific execution rather than a
    generalizable behavioral signature."""
    counts = full["MalwareFamily"].value_counts()
    rare = counts[counts < min_samples]
    if rare.empty:
        return full
    print(f"[INFO] Merging {len(rare)} rare class(es) into {bucket_name!r} "
          f"(< {min_samples} executions each): {rare.to_dict()}  "
          f"({int(rare.sum())} rows total)")
    full = full.copy()
    full.loc[full["MalwareFamily"].isin(rare.index), "MalwareFamily"] = bucket_name
    return full


def load_full_aligned_dataset() -> Tuple[pd.DataFrame, List[Dict]]:
    shared_numeric_cols = compute_shared_numeric_columns()
    frames, coverage_list = [], []
    for arch in ARCHS:
        print(f"[INFO] Aligning {arch} ...")
        merged, coverage = build_aligned_arch_dataset(arch, shared_numeric_cols)
        frames.append(merged)
        coverage_list.append(coverage)
    full = pd.concat(frames, ignore_index=True)

    # global post-aggregation low-variance sweep (see note in
    # build_aligned_arch_dataset for why this must run AFTER concatenation,
    # not per-architecture): a column is dropped only if it is constant
    # across ALL architectures' aligned rows combined.
    feat_cols_all = [c for c in full.columns if "__" in c]
    nunique = full[feat_cols_all].nunique(dropna=False)
    dropped_post_agg = nunique[nunique <= 1].index.tolist()
    if dropped_post_agg:
        print(f"[INFO] Dropping {len(dropped_post_agg)} globally-constant post-aggregation "
              f"column(s): {dropped_post_agg[:10]}{'...' if len(dropped_post_agg) > 10 else ''}")
        full = full.drop(columns=dropped_post_agg)

    full = impute_residual_nans(full, min_valid_for_mean=MIN_VALID_FOR_MEAN_IMPUTE)
    remaining_nan = int(full[[c for c in full.columns if "__" in c]].isna().sum().sum())
    if remaining_nan:
        # Should be unreachable given the imputation above always has a 0.0
        # fallback - if this ever fires, something upstream changed in a way
        # that isn't covered by the cases this function anticipates.
        raise AssertionError(f"{remaining_nan} NaN(s) remain after imputation - please report this as a bug.")

    full = merge_rare_labels(full, min_samples=MIN_SAMPLES_PER_CLASS, bucket_name=RARE_LABEL_BUCKET_NAME)

    if POST_ALIGNMENT_PER_CLASS_CAP is not None:
        capped = []
        for label, grp in full.groupby("MalwareFamily"):
            n = min(len(grp), POST_ALIGNMENT_PER_CLASS_CAP)
            capped.append(grp.sample(n, random_state=SEED))
        n_before = len(full)
        full = pd.concat(capped, ignore_index=True)
        print(f"[INFO] POST_ALIGNMENT_PER_CLASS_CAP={POST_ALIGNMENT_PER_CLASS_CAP}: "
              f"{n_before} -> {len(full)} rows")

    return full, coverage_list

## Exploratory data analysis

Descriptive statistics of the aligned dataset: class balance, missingness, skew, and feature correlation. No parameters are fitted here.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — EDA
# ═══════════════════════════════════════════════════════════

def get_modality_cols(df: pd.DataFrame) -> Dict[str, List[str]]:
    return {mod: [c for c in df.columns if c.startswith(f"{mod}__")] for mod in MODALITIES}


def eda_report(full: pd.DataFrame):
    mod_cols = get_modality_cols(full)

    # class balance
    plt.figure(figsize=(9, 5))
    counts = full["MalwareFamily"].value_counts()
    sns.barplot(x=counts.index, y=counts.values, palette="viridis")
    plt.xticks(rotation=45, ha='right')
    plt.ylabel("Aligned executions"); plt.title("Class Distribution (aligned executions)", fontweight='bold')
    plt.tight_layout(); plt.savefig(OUT_DIR / "plots" / "eda_class_distribution.png", dpi=200); plt.close()

    # arch x class heatmap
    ct = pd.crosstab(full["Arch"], full["MalwareFamily"])
    plt.figure(figsize=(10, 4))
    sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu')
    plt.title("Executions per Architecture x Class", fontweight='bold')
    plt.tight_layout(); plt.savefig(OUT_DIR / "plots" / "eda_arch_class_heatmap.png", dpi=200); plt.close()

    # missingness
    miss_pct = full[[c for cols in mod_cols.values() for c in cols]].isna().mean().sort_values(ascending=False)
    print("\n[EDA] Top missing-value columns (should be ~0 after aggregation):")
    print(miss_pct.head(10))

    # raw skew per modality (descriptive only - the fitted transform happens on TRAIN in Cell 10)
    skew_rows = []
    for mod, cols in mod_cols.items():
        for c in cols:
            vals = full[c].dropna().to_numpy()
            if vals.std() < 1e-10:
                continue
            skew_rows.append({"modality": mod, "feature": c, "skew": scipy_skew(vals)})
    skew_df = pd.DataFrame(skew_rows).sort_values("skew", key=abs, ascending=False)
    print(f"\n[EDA] Most skewed features (full data, descriptive only):")
    print(skew_df.head(10).to_string(index=False))
    skew_df.to_csv(OUT_DIR / "reports" / "eda_skew_report.csv", index=False)

    # correlation heatmap per modality (first 40 cols, for readability)
    for mod, cols in mod_cols.items():
        if not cols:
            continue
        sub_cols = cols[:40]
        corr = full[sub_cols].corr()
        plt.figure(figsize=(8, 6))
        sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, xticklabels=False, yticklabels=False, square=True)
        plt.title(f"Feature Correlation - {mod} (aggregated)", fontweight='bold')
        plt.tight_layout(); plt.savefig(OUT_DIR / "plots" / f"eda_corr_{mod.lower()}.png", dpi=200); plt.close()

    return skew_df

## Run: schema diagnostics

In [ ]:
schema_report = schema_diagnostics()
schema_report

## Run: build the aligned dataset

In [ ]:
full_df, coverage_list = load_full_aligned_dataset()
print_coverage_report(coverage_list)
print("\nFinal aligned dataset shape:", full_df.shape)
full_df.head()

## Run: exploratory data analysis

In [ ]:
skew_report = eda_report(full_df)

## Save the aligned dataset

Writes `aligned_dataset.parquet` and the alignment coverage report, which `02_camra_pipeline.ipynb` loads.

In [ ]:
ALIGNED_DATASET_PATH = OUT_DIR / "aligned_dataset.parquet"
full_df.to_parquet(ALIGNED_DATASET_PATH, index=False)

with open(OUT_DIR / "reports" / "alignment_coverage.json", "w") as f:
    json.dump(coverage_list, f, indent=2, default=str)

print(f"[INFO] Saved aligned dataset -> {ALIGNED_DATASET_PATH}  (shape={full_df.shape})")
print(f"[INFO] Saved coverage report -> {OUT_DIR / 'reports' / 'alignment_coverage.json'}")
print(f"[INFO] Saved EDA skew report -> {OUT_DIR / 'reports' / 'eda_skew_report.csv'}")
print("\n[DONE] Open Notebook 2 (preprocessing + models) to continue.")

## Raw label counts and Fig. 2 sunburst

Counts raw rows per architecture, modality, and label (reading only the label column), caches them to `reports/raw_family_counts.csv`, and draws the per-architecture sunburst charts. Needs only the configuration and helper cells. Delete the CSV to force a recount.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — FIG. 2: PER-ARCHITECTURE SUNBURST (raw captures, before curation)
# Inner ring: modality. Outer ring: label within that modality.
# Segment angles use log10(row count) so the small families stay visible.
# Counts are cached to reports/raw_family_counts.csv; re-plotting does not re-read the parquet files.
# ═══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb

RAW_COUNTS_CSV = OUT_DIR / "reports" / "raw_family_counts.csv"

def count_raw_labels() -> pd.DataFrame:
    """Row count per (architecture, modality, label) in the raw parquet files.
    Reads only the target column, batch by batch, so memory stays bounded."""
    rows = []
    for arch in ARCHS:
        base = arch_base_dir(arch)
        for mod in MODALITIES:
            path = find_modality_path(base, mod) if base is not None else None
            if path is None:
                print(f"[WARN] missing {arch}/{mod}"); continue
            pf = pq.ParquetFile(path)
            target = resolve_column([f.name for f in pf.schema_arrow], TARGET_COL_CANDIDATES, TARGET_COL_OVERRIDE)
            vc = pd.Series(dtype="int64")
            for batch in pf.iter_batches(columns=[target], batch_size=1_000_000):
                s = batch.column(0).to_pandas().astype(str).str.strip()
                vc = vc.add(s.value_counts(), fill_value=0)
            for label, n in vc.items():
                rows.append({"arch": arch, "modality": mod, "label": label, "rows": int(n)})
            print(f"[INFO] {arch}/{mod}: {int(vc.sum()):,} rows, {len(vc)} labels")
    df = pd.DataFrame(rows)
    df.to_csv(RAW_COUNTS_CSV, index=False)
    return df

raw_counts = pd.read_csv(RAW_COUNTS_CSV) if RAW_COUNTS_CSV.exists() else count_raw_labels()

# ---------------------------------------------------------------- style (matches the results figures)
plt.rcParams.update({"font.family": "serif", "font.serif": ["DejaVu Serif"], "font.size": 8,
                     "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02})
MOD_COLOUR = {"STRACE": "#009E73", "PCAP": "#D55E00", "SAR": "#E69F00"}     # Okabe-Ito green / vermillion / orange
MOD_TEXT   = {"STRACE": "white",   "PCAP": "white",   "SAR": "black"}
MOD_ORDER  = ["STRACE", "PCAP", "SAR"]
# one fixed label order for every architecture, so the four panels are directly comparable
LABEL_ORDER = (raw_counts.groupby("label")["rows"].sum().sort_values(ascending=False).index.tolist())

def _tint(colour, amount):
    r, g, b = to_rgb(colour)
    return (r + (1 - r) * amount, g + (1 - g) * amount, b + (1 - b) * amount)

def _radial_text(ax, theta_deg, r, text, colour, size, weight="normal", tangential=False):
    """Text along the radius (or along the ring if tangential=True), kept upright."""
    base = theta_deg - 90 if tangential else theta_deg
    a = (base + 180) % 360 - 180
    rot = base if -90 <= a <= 90 else base + 180
    t = np.deg2rad(theta_deg)
    ax.text(r * np.cos(t), r * np.sin(t), text, rotation=rot, rotation_mode="anchor",
            ha="center", va="center", fontsize=size, color=colour, fontweight=weight)

def plot_sunburst(arch: str, fname: str):
    sub = raw_counts[raw_counts["arch"] == arch]
    inner_sizes, inner_cols, outer_sizes, outer_cols, outer_labels, outer_mod = [], [], [], [], [], []
    for mod in MOD_ORDER:
        m = sub[sub["modality"] == mod].set_index("label")["rows"]
        labs = [l for l in LABEL_ORDER if m.get(l, 0) > 0]
        w = [np.log10(m[l] + 1) for l in labs]                     # log-scaled segment size
        inner_sizes.append(sum(w)); inner_cols.append(MOD_COLOUR[mod])
        outer_sizes += w; outer_labels += labs; outer_mod += [mod] * len(labs)
        outer_cols += [_tint(MOD_COLOUR[mod], 0.12)] * len(labs)

    fig, ax = plt.subplots(figsize=(3.2, 3.2))
    edge = dict(edgecolor="white", linewidth=1.0)
    start = 90                                                    # first modality starts at 12 o'clock, clockwise
    ax.pie(inner_sizes, radius=0.60, colors=inner_cols, startangle=start, counterclock=False,
           wedgeprops=dict(width=0.28, **edge))
    ax.pie(outer_sizes, radius=1.0, colors=outer_cols, startangle=start, counterclock=False,
           wedgeprops=dict(width=0.40, **edge))

    # modality labels (inner ring)
    tot, acc = sum(inner_sizes), 0.0
    for mod, sz in zip(MOD_ORDER, inner_sizes):
        mid = start - 360 * (acc + sz / 2) / tot; acc += sz
        _radial_text(ax, mid, 0.46, mod, MOD_TEXT[mod], 6.5, "bold", tangential=True)
    # label names (outer ring)
    tot, acc = sum(outer_sizes), 0.0
    for lab, mod, sz in zip(outer_labels, outer_mod, outer_sizes):
        mid = start - 360 * (acc + sz / 2) / tot; acc += sz
        _radial_text(ax, mid, 0.80, lab, MOD_TEXT[mod], 4.8 if 360 * sz / tot > 9 else 4.2)
    ax.set(aspect="equal")
    fig.savefig(OUT_DIR / "plots" / f"{fname}.pdf"); fig.savefig(OUT_DIR / "plots" / f"{fname}.png")
    plt.show(); plt.close(fig)
    print("saved", fname)

for arch in ARCHS:
    plot_sunburst(arch, arch.lower())          # arm.pdf, mips.pdf, mipsel.pdf, x86.pdf (the tex paths)
